# sqd-go — Minimal ERC-20 event demo

The fastest way to see `sqd-go` work: index raw **ERC-20 `Transfer`** events into ClickHouse
with nothing but a `config.yaml` — no Go code, no custom processor.

We index [LBTC](https://etherscan.io/token/0x8236a87084f8B84306f72007F36F2618A5634494)
(a standard ERC-20 token) over a small block range. End to end this takes a couple of minutes.

> Want to **derive state** (per-address balances, PnL, ...)? See `quickstart.ipynb` for the
> full custom-processor walkthrough.

## 1. Install Go

`sqd-go` is a Go program **and** a code generator: it generates Go for your indexer and
compiles it, so a working Go toolchain is required. This installs Go into `/root/.go`.

In [ ]:
# Install Go (~30s)
!wget -q -O - https://raw.githubusercontent.com/canha/golang-tools-install-script/master/goinstall.sh | bash >/dev/null 2>&1

In [ ]:
import os
# Make Go and any `go install`-ed binaries (like sqd-go) visible to every shell cell.
os.environ['GOROOT'] = '/root/.go'
os.environ['GOPATH'] = '/root/go'
os.environ['PATH']   = '/root/.go/bin:/root/go/bin:' + os.environ['PATH']
!go version

## 2. Install the sqd-go CLI

The install script runs `go install github.com/franz101/sqd-go@latest`, dropping the
`sqd-go` binary into `$(go env GOPATH)/bin` (already on `PATH` from the cell above).

In [ ]:
# Install the CLI (first run compiles dependencies, ~1-2 min)
!curl -sSL https://raw.githubusercontent.com/franz101/sqd-go/main/install.sh | bash
!sqd-go help | head -20

## 3. Run ClickHouse

sqd-go indexes into [ClickHouse](https://clickhouse.com). In Colab we use the official
single binary with an **empty password** on the default ports (native `9000`, HTTP `8123`).

In [ ]:
# Download the ClickHouse single binary
!curl -s https://clickhouse.com/ | sh >/dev/null 2>&1

# Start the server in the background (empty password) and give it a moment to come up
import subprocess, time
subprocess.Popen(['./clickhouse', 'server'],
                 stdout=open('clickhouse-server.log','w'),
                 stderr=subprocess.STDOUT)
time.sleep(8)
!./clickhouse client --query "SELECT 'ClickHouse is up' AS status"

## 4. Create the project

A sqd-go project is just a directory with a `config.yaml`. The config names the chain,
the block range, the contract address, and which event signatures to decode.

In [ ]:
!mkdir -p erc20demo

In [ ]:
%%writefile erc20demo/config.yaml
name: erc20demo
chains:
  - id: 1
    start_block: 20600000
    end_block: 20601000          # ~1000 blocks; ~86 LBTC transfers
    contracts:
      - name: LBTC
        address: "0x8236a87084f8B84306f72007F36F2618A5634494"
        events:
          - event: Transfer(address indexed from, address indexed to, uint256 value)

**Config fields**

- `name` — the ClickHouse database the data lands in.
- `chains[].id` — EVM chain id (1 = Ethereum mainnet).
- `start_block` / `end_block` — the (inclusive) range to backfill.
- `contracts[].address` — the contract to watch.
- `events[]` — Solidity event signatures; `indexed` marks topic args. Each becomes a typed
  ClickHouse table named `<contract>_<event>_events` (here `lbtc_transfer_events`).

## 5. Index

Point the CLI at the project. `--parallel-fetch` backfills with several range workers;
`--restart` drops any previous data for this project first.

> **Flags are space-separated.** Use `--start-block 20600000`, *not* `--start-block=20600000`
> (the `=` form is silently ignored). The ClickHouse connection is passed via env vars.

In [ ]:
!CLICKHOUSE_HOST=127.0.0.1 \
 CLICKHOUSE_NATIVE_PORT=9000 CLICKHOUSE_HTTP_PORT=8123 \
 CLICKHOUSE_USER=default CLICKHOUSE_PASSWORD='' \
 sqd-go start erc20demo --start-block 20600000 --end-block 20601000 --restart --parallel-fetch

## 6. Query the data

The decoded events are now a regular ClickHouse table you can query with SQL.

In [ ]:
!./clickhouse client --query "SHOW TABLES FROM erc20demo"

In [ ]:
!./clickhouse client --query "SELECT count() AS transfers FROM erc20demo.lbtc_transfer_events"

In [ ]:
# Each row carries the decoded args (from/to/value) plus always-present metadata.
# (from/to/value are SQL keywords, so they're backtick-quoted.)
!./clickhouse client --query "SELECT block_number, transaction_index, log_index, concat('0x', lower(hex(\`from\`))) AS from_addr, concat('0x', lower(hex(\`to\`))) AS to_addr, \`value\` FROM erc20demo.lbtc_transfer_events ORDER BY block_number, log_index LIMIT 5 FORMAT PrettyCompact"

## What you got

- `lbtc_transfer_events` — one row per decoded `Transfer`, with the event args plus the
  **always-present** fields `block_number`, `block_timestamp`, `transaction_index`, `log_index`
  (see [`docs/EVENT_FIELDS.md`](EVENT_FIELDS.md)).
- `sync_state` — the indexer checkpoint (so re-runs resume instead of restarting).

### Next steps

- **Derive state** (balances, counts, PnL) with a custom processor → `quickstart.ipynb`.
- More events: add lines under `events:` and re-run.
- Observability & tuning: [`docs/METRICS.md`](METRICS.md).